<a href="https://colab.research.google.com/github/minyi-k03/Large-Language-Model-LLM-/blob/Fine-Tuning/Llama3_1_%ED%8A%B9%ED%97%88_%EC%B9%B4%ED%85%8C%EA%B3%A0%EB%A6%AC_%EB%B6%84%EB%A5%98_%ED%94%84%EB%A1%AC%ED%94%84%ED%8A%B8_%ED%8A%9C%EB%8B%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Llama 3.1을 이용해서 특허 카테고리 자동 분류하기 - 프롬프트 튜닝을 통한 성능 개선
## 작성자 : AISchool ( http://aischool.ai/%ec%98%a8%eb%9d%bc%ec%9d%b8-%ea%b0%95%ec%9d%98-%ec%b9%b4%ed%85%8c%ea%b3%a0%eb%a6%ac/ )
## Reference : https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct
## [특허 분야 자동분류 데이터] 데이터 : https://www.aihub.or.kr/aihubdata/data/view.do?currMenu=115&topMenu=100&dataSetSn=547

In [1]:
!nvidia-smi

Sun Jan 25 06:35:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# 라이브러리 설치

In [2]:
import torch

# 1. GPU 확인 (잘 잡혀있는지 체크)
if torch.cuda.is_available():
    print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("GPU가 없습니다. 상단 메뉴 [런타임] -> [런타임 유형 변경]에서 T4 GPU를 선택하세요.")

print("\nInstalling Libraries for Llama-3.1...")

# 2. 필수 라이브러리 설치
!pip install -U "transformers>=4.43.0" "accelerate>=0.26.0" "bitsandbytes>=0.42.0" "huggingface_hub"

print("\nSetup Completed.")

GPU Detected: Tesla T4

Installing Libraries for Llama-3.1...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 14.3 MB/s eta 0:00:00

Setup Completed.


In [3]:
import json
import pandas as pd

## [특허 분야 자동분류 데이터] 데이터 불러오기

In [4]:
!unzip Patent_AutoMatic_Regression.zip

Archive:  Patent_AutoMatic_Regression.zip
   creating: 라벨링데이터/
   creating: 라벨링데이터/A_농업_임업및어업_01_03/
   creating: 라벨링데이터/A_농업_임업및어업_01_03/01_농업/
   creating: 라벨링데이터/A_농업_임업및어업_01_03/01_농업/011_작물재배업/
  inflating: 라벨링데이터/A_농업_임업및어업_01_03/01_농업/011_작물재배업/01110_곡물및기타식량작물재배업.json  
  inflating: 라벨링데이터/A_농업_임업및어업_01_03/01_농업/011_작물재배업/01121_채소작물재배업.json  
  inflating: 라벨링데이터/A_농업_임업및어업_01_03/01_농업/011_작물재배업/01122_화훼작물재배업.json  
  inflating: 라벨링데이터/A_농업_임업및어업_01_03/01_농업/011_작물재배업/01123_종자및묘목생산업.json  
  inflating: 라벨링데이터/A_농업_임업및어업_01_03/01_농업/011_작물재배업/01131_과실작물재배업.json  
  inflating: 라벨링데이터/A_농업_임업및어업_01_03/01_농업/011_작물재배업/01132_음료용및향신용작물재배업.json  
  inflating: 라벨링데이터/A_농업_임업및어업_01_03/01_농업/011_작물재배업/01140_기타작물재배업.json  
  inflating: 라벨링데이터/A_농업_임업및어업_01_03/01_농업/011_작물재배업/01151_콩나물재배업.json  
  inflating: 라벨링데이터/A_농업_임업및어업_01_03/01_농업/011_작물재배업/01159_기타시설작물재배업.json  
   creating: 라벨링데이터/A_농업_임업및어업_01_03/01_농업/012_축산업/
  inflating: 라벨링데이터/A_농업_임업및어업_01_03/01_농업/012_축산업/01211_젖소사육업.json  
 

## 원천 데이터 불러오기

In [5]:
# 파일 경로
file_path = '/content/원천데이터/A_농업_임업및어업_01_03/01_농업/011_작물재배업/01110_곡물및기타식량작물재배업.json'

# 파일 열기 및 JSON 읽기
with open(file_path, 'r', encoding='utf-8') as file:
    raw_data = json.load(file)
raw_data = raw_data['dataset']

In [6]:
raw_data

[{'application_year': '2021',
  'invention_title': '기능성이 향상된 메밀싹 및 새싹채소의 재배방법',
  'ipc_section': 'A',
  'register_date': '20210927',
  'abstract': '본 발명은 약용식물을 선별하고 가공하는 단계; 상기 가공한 약용식물을 혼합한 후 추출 및 희석하여 약용식물 희석액을 제조하는 단계; 및 메밀종자를 파종한 후, 파종 상토에 상기 제조한 약용식물 희석액을 분무하면서 특정 온도 및 광조건에서 재배하는 단계를 포함하는 메밀싹의 재배방법 및 상기 방법으로 재배한 메밀싹에 관한 것이다.',
  'ipc_subclass': 'A01G',
  'ipc_main': 'A01G-022/20',
  'register_year': '2021',
  'ipc_all': 'A01G-022/20||A01G-007/06||A01N-025/02||A01N-065/08||A01N-065/30||A01N-065/34||A23L-033/00',
  'ipc_class': 'A01',
  'claims': '(1) 녹차, 짚신나물, 마디풀 및 소리쟁이를 각각 증열처리하고 건조하여 건조 녹차, 건조 짚신나물, 건조 마디풀 및 건조 소리쟁이를 제조하는 단계;(2) 상기 (1)단계의 제조한 건조 녹차, 건조 짚신나물, 건조 마디풀 및 건조 소리쟁이와 커피 분말을 혼합하여 약용식물 혼합물을 제조하는 단계;(3) 상기 (2)단계의 제조한 약용식물 혼합물에 물을 첨가하여 추출한 후 여과하여 약용식물 혼합 추출액을 제조하는 단계;(4) 상기 (3)단계의 제조한 약용식물 혼합 추출액에 물을 첨가하여 약용식물 희석액을 제조하는 단계; 및(5) 메밀종자를 파종한 후, 파종 상토에 상기 (4)단계의 제조한 약용식물 희석액을 분무하면서 재배하는 단계를 포함하는 메밀싹의 재배방법.제1항 내지 제3항 중 어느 한 항의 방법으로 재배된 메밀싹.',
  'application_date': '20210823',
  

In [7]:
len(raw_data)

50

In [8]:
raw_data[0]

{'application_year': '2021',
 'invention_title': '기능성이 향상된 메밀싹 및 새싹채소의 재배방법',
 'ipc_section': 'A',
 'register_date': '20210927',
 'abstract': '본 발명은 약용식물을 선별하고 가공하는 단계; 상기 가공한 약용식물을 혼합한 후 추출 및 희석하여 약용식물 희석액을 제조하는 단계; 및 메밀종자를 파종한 후, 파종 상토에 상기 제조한 약용식물 희석액을 분무하면서 특정 온도 및 광조건에서 재배하는 단계를 포함하는 메밀싹의 재배방법 및 상기 방법으로 재배한 메밀싹에 관한 것이다.',
 'ipc_subclass': 'A01G',
 'ipc_main': 'A01G-022/20',
 'register_year': '2021',
 'ipc_all': 'A01G-022/20||A01G-007/06||A01N-025/02||A01N-065/08||A01N-065/30||A01N-065/34||A23L-033/00',
 'ipc_class': 'A01',
 'claims': '(1) 녹차, 짚신나물, 마디풀 및 소리쟁이를 각각 증열처리하고 건조하여 건조 녹차, 건조 짚신나물, 건조 마디풀 및 건조 소리쟁이를 제조하는 단계;(2) 상기 (1)단계의 제조한 건조 녹차, 건조 짚신나물, 건조 마디풀 및 건조 소리쟁이와 커피 분말을 혼합하여 약용식물 혼합물을 제조하는 단계;(3) 상기 (2)단계의 제조한 약용식물 혼합물에 물을 첨가하여 추출한 후 여과하여 약용식물 혼합 추출액을 제조하는 단계;(4) 상기 (3)단계의 제조한 약용식물 혼합 추출액에 물을 첨가하여 약용식물 희석액을 제조하는 단계; 및(5) 메밀종자를 파종한 후, 파종 상토에 상기 (4)단계의 제조한 약용식물 희석액을 분무하면서 재배하는 단계를 포함하는 메밀싹의 재배방법.제1항 내지 제3항 중 어느 한 항의 방법으로 재배된 메밀싹.',
 'application_date': '20210823',
 'register_num

In [9]:
raw_data[0].keys()

dict_keys(['application_year', 'invention_title', 'ipc_section', 'register_date', 'abstract', 'ipc_subclass', 'ipc_main', 'register_year', 'ipc_all', 'ipc_class', 'claims', 'application_date', 'register_number', 'documentId', 'application_number'])

In [10]:
def extract_fields(data):
    # 필요한 필드를 추출
    invention_title = data.get('invention_title', '')
    abstract = data.get('abstract', '')
    claims = data.get('claims', '')

    # 추출한 내용을 하나의 문자열로 연결
    result = f"invention_title: {invention_title} abstract: {abstract} claims: {claims}"

    return result

In [11]:
data_0 = extract_fields(raw_data[0])
data_0

'invention_title: 기능성이 향상된 메밀싹 및 새싹채소의 재배방법 abstract: 본 발명은 약용식물을 선별하고 가공하는 단계; 상기 가공한 약용식물을 혼합한 후 추출 및 희석하여 약용식물 희석액을 제조하는 단계; 및 메밀종자를 파종한 후, 파종 상토에 상기 제조한 약용식물 희석액을 분무하면서 특정 온도 및 광조건에서 재배하는 단계를 포함하는 메밀싹의 재배방법 및 상기 방법으로 재배한 메밀싹에 관한 것이다. claims: (1) 녹차, 짚신나물, 마디풀 및 소리쟁이를 각각 증열처리하고 건조하여 건조 녹차, 건조 짚신나물, 건조 마디풀 및 건조 소리쟁이를 제조하는 단계;(2) 상기 (1)단계의 제조한 건조 녹차, 건조 짚신나물, 건조 마디풀 및 건조 소리쟁이와 커피 분말을 혼합하여 약용식물 혼합물을 제조하는 단계;(3) 상기 (2)단계의 제조한 약용식물 혼합물에 물을 첨가하여 추출한 후 여과하여 약용식물 혼합 추출액을 제조하는 단계;(4) 상기 (3)단계의 제조한 약용식물 혼합 추출액에 물을 첨가하여 약용식물 희석액을 제조하는 단계; 및(5) 메밀종자를 파종한 후, 파종 상토에 상기 (4)단계의 제조한 약용식물 희석액을 분무하면서 재배하는 단계를 포함하는 메밀싹의 재배방법.제1항 내지 제3항 중 어느 한 항의 방법으로 재배된 메밀싹.'

## 라벨링 데이터 불러오기

In [12]:
# 파일 경로
file_path = '/content/라벨링데이터/A_농업_임업및어업_01_03/01_농업/011_작물재배업/01110_곡물및기타식량작물재배업.json'

# 파일 열기 및 JSON 읽기
with open(file_path, 'r', encoding='utf-8') as file:
    label_data = json.load(file)
label_data = label_data['dataset']

In [13]:
label_data

[{'updateDate': '20220608',
  'LLno': 'A',
  'SSno': '01110',
  'Stext': '곡물 및 기타 식량작물 재배업',
  'SStext': '곡물 및 기타 식량작물 재배업',
  'Mtext': '작물 재배업',
  'Lno': '01',
  'ipc_main': 'A01G-022/20',
  'Mno': '011',
  'country_code': 'KR',
  'Sno': '0111',
  'LLtext': '농업, 임업 및 어업(01~03)',
  'documentId': 'kr20210110983b1',
  'application_number': '1020210110983',
  'Ltext': '농업',
  'keyword': '노지재배/메밀노지재배/메밀/파종/특정온도/약용식물/방법/증열/커피분말/추출액',
  'document_type': 'B1'},
 {'updateDate': '20220608',
  'LLno': 'A',
  'SSno': '01110',
  'Stext': '곡물 및 기타 식량작물 재배업',
  'SStext': '곡물 및 기타 식량작물 재배업',
  'Mtext': '작물 재배업',
  'Lno': '01',
  'ipc_main': 'A01G-031/02',
  'Mno': '011',
  'country_code': 'KR',
  'Sno': '0111',
  'LLtext': '농업, 임업 및 어업(01~03)',
  'documentId': 'kr20210041526b1',
  'application_number': '1020210041526',
  'Ltext': '농업',
  'keyword': '노지재배/뿌리작물노지재배/수경재배/양액/밸브/배출/방법/이온/흡수율/괴리율',
  'document_type': 'B1'},
 {'updateDate': '20220608',
  'LLno': 'A',
  'SSno': '01110',
  'Stext': '곡물 및 기타 식

In [14]:
len(label_data)

50

In [15]:
label_data[0]

{'updateDate': '20220608',
 'LLno': 'A',
 'SSno': '01110',
 'Stext': '곡물 및 기타 식량작물 재배업',
 'SStext': '곡물 및 기타 식량작물 재배업',
 'Mtext': '작물 재배업',
 'Lno': '01',
 'ipc_main': 'A01G-022/20',
 'Mno': '011',
 'country_code': 'KR',
 'Sno': '0111',
 'LLtext': '농업, 임업 및 어업(01~03)',
 'documentId': 'kr20210110983b1',
 'application_number': '1020210110983',
 'Ltext': '농업',
 'keyword': '노지재배/메밀노지재배/메밀/파종/특정온도/약용식물/방법/증열/커피분말/추출액',
 'document_type': 'B1'}

In [16]:
label_data[0].keys()

dict_keys(['updateDate', 'LLno', 'SSno', 'Stext', 'SStext', 'Mtext', 'Lno', 'ipc_main', 'Mno', 'country_code', 'Sno', 'LLtext', 'documentId', 'application_number', 'Ltext', 'keyword', 'document_type'])

In [17]:
label_data[0]['Ltext']

'농업'

# Llama 3.1 모델 불러오기

In [18]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


# 1. Hugging Face Token 설정
os.environ['HF_TOKEN'] = "Input Your Token"


# 2. 모델 설정 및 로드
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# [수정 1] T4 GPU 메모리(16GB) 초과 방지를 위한 4-bit 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16 # T4는 float16 권장
)

print(f" Loading Model: {model_id}...")

# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Llama 모델 에러 방지용 설정 (Padding Token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# [수정 2] 모델 로드 (bfloat16 제거 -> float16 & 4bit 적용)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config, # 4-bit 양자화 적용
    device_map="auto",
    torch_dtype=torch.float16       # T4 호환성 맞춤
)

print(f"Model Loaded Successfully on {model.device}")

 Loading Model: meta-llama/Meta-Llama-3.1-8B-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model Loaded Successfully on cuda:0


In [20]:
def generate_response(system_message, user_message, tokenizer, model):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    outputs = model.generate(
        input_ids,
        max_new_tokens=256,
        eos_token_id=terminators,
        do_sample=False,   # 항상 가장 확률이 높은 값으로 예측
        temperature=0.6,
        top_p=0.9
    )
    response = outputs[0][input_ids.shape[-1]:]

    return tokenizer.decode(response, skip_special_tokens=True)

In [21]:
print(model.device)

cuda:0


# 특허 카테고리 분류 테스트

In [ ]:
# 특허_카테고리_list = ['임업',
#  '어업',
#  '농업',
#  '자동차 및 트레일러 제조업',
#  '인쇄 및 기록매체 복제업',
#  '펄프, 종이 및 종이제품 제조업',
#  '비금속 광물제품 제조업',
#  '고무 및 플라스틱제품 제조업',
#  '코크스, 연탄 및 석유정제품 제조업',
#  '음료 제조업',
#  '의복, 의복 액세서리 및 모피제품 제조업',
#  '가구 제조업',
#  '기타 운송장비 제조업',
#  '전기장비 제조업',
#  '의료용 물질 및 의약품 제조업',
#  '기타 제품 제조업',
#  '산업용 기계 및 장비 수리업',
#  '화학 물질 및 화학제품 제조업; 의약품 제외',
#  '목재 및 나무제품 제조업; 가구 제외',
#  '전자 부품, 컴퓨터, 영상, 음향 및 통신장비 제조업',
#  '섬유제품 제조업; 의복 제외',
#  '담배 제조업',
#  '의료, 정밀, 광학 기기 및 시계 제조업',
#  '금속 가공제품 제조업; 기계 및 가구 제외',
#  '기타 기계 및 장비 제조업',
#  '가죽, 가방 및 신발 제조업',
#  '1차 금속 제조업',
#  '식료품 제조업',
#  '금속 광업',
#  '광업 지원 서비스업',
#  '석탄, 원유 및 천연가스 광업',
#  '비금속광물 광업; 연료용 제외']

In [23]:
특허_카테고리_list = ['임업',
 '어업',
 '농업']

In [ ]:
len(특허_카테고리_list)

3

In [24]:
#System Prompt Prompt Tuning 전 기본 프롬프트
system_prompt = f"너는 특허 카테고리를 분류하는 전문가야. \
아래 내용을 다음 특허 카테고리 중 하나로 분류해줘. 가능한 특허 카테고리 : {특허_카테고리_list}. \
최종 출력 결과는 다른말은 하지말고 분류한 카테고리만 출력해줘."
system_prompt

"너는 특허 카테고리를 분류하는 전문가야. 아래 내용을 다음 특허 카테고리 중 하나로 분류해줘. 가능한 특허 카테고리 : ['임업', '어업', '농업']. 최종 출력 결과는 다른말은 하지말고 분류한 카테고리만 출력해줘."

In [25]:
data_0

'invention_title: 기능성이 향상된 메밀싹 및 새싹채소의 재배방법 abstract: 본 발명은 약용식물을 선별하고 가공하는 단계; 상기 가공한 약용식물을 혼합한 후 추출 및 희석하여 약용식물 희석액을 제조하는 단계; 및 메밀종자를 파종한 후, 파종 상토에 상기 제조한 약용식물 희석액을 분무하면서 특정 온도 및 광조건에서 재배하는 단계를 포함하는 메밀싹의 재배방법 및 상기 방법으로 재배한 메밀싹에 관한 것이다. claims: (1) 녹차, 짚신나물, 마디풀 및 소리쟁이를 각각 증열처리하고 건조하여 건조 녹차, 건조 짚신나물, 건조 마디풀 및 건조 소리쟁이를 제조하는 단계;(2) 상기 (1)단계의 제조한 건조 녹차, 건조 짚신나물, 건조 마디풀 및 건조 소리쟁이와 커피 분말을 혼합하여 약용식물 혼합물을 제조하는 단계;(3) 상기 (2)단계의 제조한 약용식물 혼합물에 물을 첨가하여 추출한 후 여과하여 약용식물 혼합 추출액을 제조하는 단계;(4) 상기 (3)단계의 제조한 약용식물 혼합 추출액에 물을 첨가하여 약용식물 희석액을 제조하는 단계; 및(5) 메밀종자를 파종한 후, 파종 상토에 상기 (4)단계의 제조한 약용식물 희석액을 분무하면서 재배하는 단계를 포함하는 메밀싹의 재배방법.제1항 내지 제3항 중 어느 한 항의 방법으로 재배된 메밀싹.'

In [27]:
#Prompt-Tuning 전에 대한 답
llama3_1_inference_result = generate_response(system_message=system_prompt,
                              user_message=data_0,
                              tokenizer=tokenizer,
                              model=model)
print(llama3_1_inference_result)
# 정답 : 농업

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


임업


In [28]:
raw_data[1]

{'application_year': '2021',
 'invention_title': '수경재배 베드 및 뿌리 생장 정보를 이용한 수경재배 방법',
 'ipc_section': 'A',
 'register_date': '20210928',
 'abstract': '본 발명은 수경재배 베드에 관한 것으로, 수경재배 장치에 사용되는 수경재배 베드에 있어서, 상기 수경재배 장치에서 양액을 공급 받고, 식물이 수용되는 베드; 상기 베드의 측면 상부에 형성되어 상기 양액을 배출하는 제1 배출구; 상기 베드의 하단에 형성되어 상기 양액을 배출하는 제2 배출구; 및 상기 제2 배출구를 통한 상기 양액의 배출 여부를 조절하는 밸브;를 포함하되, 상기 수경재배 장치가 상기 식물의 뿌리 생장을 판단하여 상기 양액이 상기 제1 배출구 또는 상기 제2 배출구를 통해 배출되도록 상기 밸브를 제어한다.',
 'ipc_subclass': 'A01G',
 'ipc_main': 'A01G-031/02',
 'register_year': '2021',
 'ipc_all': 'A01G-031/02||A01G-024/40||A01G-007/00||G06Q-050/02',
 'ipc_class': 'A01',
 'claims': '수경재배 장치에 사용되는 수경재배 베드에 있어서, 상기 수경재배 장치에서 양액을 공급 받고, 식물이 수용되는 베드; 상기 베드의 측면 상부에 형성되어 상기 양액을 배출하는 제1 배출구; 상기 베드의 하단에 형성되어 상기 양액을 배출하는 제2 배출구; 및 상기 제2 배출구를 통한 상기 양액의 배출 여부를 조절하는 밸브;를 포함하되, 상기 수경재배 장치가 상기 식물의 뿌리 생장을 판단하여 상기 양액이 상기 제1 배출구 또는 상기 제2 배출구를 통해 배출되도록 상기 밸브를 제어하고, 상기 뿌리 생장의 판단은, 상기 수경재배 장치가 상기 제1 배출구를 통해 배출될 때의 상기 양액 및 상기 제2 배출구를 통해 배출될 때의 상기 양액에 대한 각각의 식물 뿌리의 이온 흡수율을 아래

In [29]:
data_1 = extract_fields(raw_data[1])
data_1

'invention_title: 수경재배 베드 및 뿌리 생장 정보를 이용한 수경재배 방법 abstract: 본 발명은 수경재배 베드에 관한 것으로, 수경재배 장치에 사용되는 수경재배 베드에 있어서, 상기 수경재배 장치에서 양액을 공급 받고, 식물이 수용되는 베드; 상기 베드의 측면 상부에 형성되어 상기 양액을 배출하는 제1 배출구; 상기 베드의 하단에 형성되어 상기 양액을 배출하는 제2 배출구; 및 상기 제2 배출구를 통한 상기 양액의 배출 여부를 조절하는 밸브;를 포함하되, 상기 수경재배 장치가 상기 식물의 뿌리 생장을 판단하여 상기 양액이 상기 제1 배출구 또는 상기 제2 배출구를 통해 배출되도록 상기 밸브를 제어한다. claims: 수경재배 장치에 사용되는 수경재배 베드에 있어서, 상기 수경재배 장치에서 양액을 공급 받고, 식물이 수용되는 베드; 상기 베드의 측면 상부에 형성되어 상기 양액을 배출하는 제1 배출구; 상기 베드의 하단에 형성되어 상기 양액을 배출하는 제2 배출구; 및 상기 제2 배출구를 통한 상기 양액의 배출 여부를 조절하는 밸브;를 포함하되, 상기 수경재배 장치가 상기 식물의 뿌리 생장을 판단하여 상기 양액이 상기 제1 배출구 또는 상기 제2 배출구를 통해 배출되도록 상기 밸브를 제어하고, 상기 뿌리 생장의 판단은, 상기 수경재배 장치가 상기 제1 배출구를 통해 배출될 때의 상기 양액 및 상기 제2 배출구를 통해 배출될 때의 상기 양액에 대한 각각의 식물 뿌리의 이온 흡수율을 아래 [수식 1]을 이용해 분석하여, 상기 분석한 각각의 식물 뿌리의 이온 흡수율의 괴리율을 바탕으로 판단하는 것 을 특징으로 하는 수경재배 베드. [수식 1] (여기서, 은 뿌리 내부의 ion이라는 이름의 이온의 시간에 따른 조성변화, 은 이온의 물리적 특성으로 는 확산 상수, M은 분자량(단원자 이온의 경우에는 원자량)이며, 는 이온의 전기적 특성, 은 Goldman의 방정식으로 계산되는 식물 뿌리와 주변 사이의 상호 작용에 대한 계수이며, 는 식물종에 대한 계수

In [30]:
label_data[1]

{'updateDate': '20220608',
 'LLno': 'A',
 'SSno': '01110',
 'Stext': '곡물 및 기타 식량작물 재배업',
 'SStext': '곡물 및 기타 식량작물 재배업',
 'Mtext': '작물 재배업',
 'Lno': '01',
 'ipc_main': 'A01G-031/02',
 'Mno': '011',
 'country_code': 'KR',
 'Sno': '0111',
 'LLtext': '농업, 임업 및 어업(01~03)',
 'documentId': 'kr20210041526b1',
 'application_number': '1020210041526',
 'Ltext': '농업',
 'keyword': '노지재배/뿌리작물노지재배/수경재배/양액/밸브/배출/방법/이온/흡수율/괴리율',
 'document_type': 'B1'}

In [31]:
#Prompt-Tuning 전에 프롬프트에 대한 답변
llama3_1_inference_result = generate_response(system_message=system_prompt,
                              user_message=data_1,
                              tokenizer=tokenizer,
                              model=model)
print(llama3_1_inference_result)
# 정답 : 농업

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


임업


# 전체 데이터 불러오기

In [32]:
def list_all_files_in_directory(directory_path):
    # 파일 경로를 저장할 리스트 초기화
    all_files = []

    # directory_path 하위 폴더의 모든 파일들의 경로를 리스트에 저장
    for root, dirs, files in os.walk(directory_path):
        for file in files:
            # 파일 경로를 리스트에 추가
            all_files.append(os.path.join(root, file))

    # 전체 파일 경로 리스트 반환
    return all_files

In [33]:
directory_path = '/content/원천데이터'
raw_file_list = list_all_files_in_directory(directory_path)
raw_file_list

['/content/원천데이터/A_농업_임업및어업_01_03/02_임업/020_임업/02012_육림업.json',
 '/content/원천데이터/A_농업_임업및어업_01_03/02_임업/020_임업/02030_임산물채취업.json',
 '/content/원천데이터/A_농업_임업및어업_01_03/02_임업/020_임업/02040_임업관련서비스업.json',
 '/content/원천데이터/A_농업_임업및어업_01_03/02_임업/020_임업/02011_임업용종묘생산업.json',
 '/content/원천데이터/A_농업_임업및어업_01_03/03_어업/031_어로어업/03111_원양어업.json',
 '/content/원천데이터/A_농업_임업및어업_01_03/03_어업/031_어로어업/03112_연근해어업.json',
 '/content/원천데이터/A_농업_임업및어업_01_03/03_어업/032_양식어업및어업관련서비스업/03211_해수면양식어업.json',
 '/content/원천데이터/A_농업_임업및어업_01_03/03_어업/032_양식어업및어업관련서비스업/03212_내수면양식어업.json',
 '/content/원천데이터/A_농업_임업및어업_01_03/03_어업/032_양식어업및어업관련서비스업/03220_어업관련서비스업.json',
 '/content/원천데이터/A_농업_임업및어업_01_03/03_어업/032_양식어업및어업관련서비스업/03213_수산물부화및수산종자생산업.json',
 '/content/원천데이터/A_농업_임업및어업_01_03/01_농업/012_축산업/01239_기타가금류및조류사육업.json',
 '/content/원천데이터/A_농업_임업및어업_01_03/01_농업/012_축산업/01231_양계업.json',
 '/content/원천데이터/A_농업_임업및어업_01_03/01_농업/012_축산업/01299_그외기타축산업.json',
 '/content/원천데이터/A_농업_임업및어업_01_03/01_농업/012_축산업/01211_젖소사육업.json',

In [34]:
len(raw_file_list)

461

In [35]:
def load_json_dataset(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        data = json.load(file)

    return data.get('dataset')

In [36]:
dataframes = []
for idx, file in enumerate(raw_file_list):
    print(idx, file)
    data = load_json_dataset(file)
    df = pd.DataFrame(data)
    dataframes.append(df)  # 리스트에 DataFrame 추가

raw_df = pd.concat(dataframes, ignore_index=True)

0 /content/원천데이터/A_농업_임업및어업_01_03/02_임업/020_임업/02012_육림업.json
1 /content/원천데이터/A_농업_임업및어업_01_03/02_임업/020_임업/02030_임산물채취업.json
2 /content/원천데이터/A_농업_임업및어업_01_03/02_임업/020_임업/02040_임업관련서비스업.json
3 /content/원천데이터/A_농업_임업및어업_01_03/02_임업/020_임업/02011_임업용종묘생산업.json
4 /content/원천데이터/A_농업_임업및어업_01_03/03_어업/031_어로어업/03111_원양어업.json
5 /content/원천데이터/A_농업_임업및어업_01_03/03_어업/031_어로어업/03112_연근해어업.json
6 /content/원천데이터/A_농업_임업및어업_01_03/03_어업/032_양식어업및어업관련서비스업/03211_해수면양식어업.json
7 /content/원천데이터/A_농업_임업및어업_01_03/03_어업/032_양식어업및어업관련서비스업/03212_내수면양식어업.json
8 /content/원천데이터/A_농업_임업및어업_01_03/03_어업/032_양식어업및어업관련서비스업/03220_어업관련서비스업.json
9 /content/원천데이터/A_농업_임업및어업_01_03/03_어업/032_양식어업및어업관련서비스업/03213_수산물부화및수산종자생산업.json
10 /content/원천데이터/A_농업_임업및어업_01_03/01_농업/012_축산업/01239_기타가금류및조류사육업.json
11 /content/원천데이터/A_농업_임업및어업_01_03/01_농업/012_축산업/01231_양계업.json
12 /content/원천데이터/A_농업_임업및어업_01_03/01_농업/012_축산업/01299_그외기타축산업.json
13 /content/원천데이터/A_농업_임업및어업_01_03/01_농업/012_축산업/01211_젖소사육업.json
14 /content/원천데이터/A_농업_

In [37]:
raw_df

,application_year,invention_title,ipc_section,register_date,abstract,ipc_subclass,ipc_main,register_year,ipc_all,ipc_class,claims,application_date,register_number,documentId,application_number,open_number,open_date,open_year,applicant_name
0,2021,수목 보호 지지대,A,20210609,"본 발명은 수목 보호 지지대에 관한 것으로, 지면에 하단이 고정되는 다수의 버팀부재...",A01G,A01G-017/14,2021,A01G-017/14||A01G-017/10||A01G-009/12,A01,"수목(100)의 둘레 면에 접촉되는 다수의 밀착부재(2)와, 상기 밀착부재(2)를 ...",20210316,102265472,kr20210033985b1,1020210033985,NaN,NaN,NaN,NaN
1,2021,나무보호장치,A,NaN,강풍에 나무들이나 전봇대들이 뽑히거나 부러지고 넘어져서 피해를 입는 사고가 많다. ...,A01G,A01G-017/14,NaN,A01G-017/14||A01G-009/12,A01,'기초보호대' 와 '땅속지지대' 를 땅속에 설치하여 강풍에 나무가 옆으로 기울거나 ...,20210309,NaN,kr20210000760u,2020210000760,2020210000706,NaN,NaN,NaN
2,2021,수목 관리 장치 및 시스템,A,20210423,"본 발명은 수목 관리 장치 및 시스템에 관한 것으로, 더욱 상세하게는 수목을 안정적...",A01G,A01G-017/14,2021,A01G-017/14||A01G-027/00||F21S-008/08||F21S-00...,A01,"수목(1)을 감싸도록 주변에 설치되어 수목(1)을 지지하고, 유수를 집수하여 수목(...",20210118,102246082,kr20210006694b1,1020210006694,NaN,NaN,NaN,NaN
3,2021,휴대용 나무 안정지지대,A,20210601,"본 발명은 휴대용 나무 안정지지대에 관한 것으로서, 회동축(A)을 중심으로 크로스 ...",A01G,A01G-023/04,2021,A01G-023/04||A01G-017/14,A01,"회동축(A)을 중심으로 크로스 결합된 것으로서, 땅(E)에 지지되어 이식 대기중인 ...",20210106,102261923,kr20210001336b1,1020210001336,NaN,NaN,NaN,NaN
4,2020,수목보호용 블록,A,20210611,본 발명은 블록의 조립에 의해 이루어진 식재 공간 내부에 식재되는 수목의 위치에 맞...,A01G,A01G-013/02,2021,A01G-013/02||C04B-020/00||C04B-028/02||E01C-00...,A01,수목의 주위에 설치되어 수목이 설치된 지반의 흙이 외부로 흩어지는 것을 방지하는 수...,20201113,102266293,kr20200152106b1,1020200152106,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23045,2019,"버클, 벨트 및 벨트 부품",A,NaN,"피복의 흘러내림을 억제하는 벨트 버클로서, 벨트의 일단이 고정된 버클 본체와, 버클...",A41F,A41F-009/00,NaN,A41F-009/00||A41F-017/02||A44B-011/00||A44B-01...,A41,"피복의 흘러내림을 억제하는 벨트 버클에 있어서,상기 벨트의 일단이 고정되는 버클 본...",20190807,NaN,kr20207036302a,1020207036302,1020210009371,NaN,NaN,NaN
23046,2019,히든 벨트,A,NaN,"본 발명은 히든벨트에 대한 것으로서, 본 발명의 일실시예에 따른 벨트는 고정부와 긴...",A41F,A41F-009/02,NaN,A41F-009/02||A41F-009/00,A41,"벨트에 있어서,길이를 조절하고 고정할 수 있는 고정부;벨트에 몸통이 되는 긴 띠;바...",20190730,NaN,kr20190092477a,1020190092477,1020210014419,NaN,NaN,NaN
23047,2019,버클 전자 장치를 포함하는 스마트 벨트,A,NaN,"본 발명의 실시 예에 따른 스마트 벨트는, 착용 부위의 복부 압력 및 자세 기울기를...",A41F,A41F-009/00,NaN,A41F-009/00||A44B-011/00||A61B-005/00||A61B-00...,A41,착용 부위의 복부 압력 및 자세 기울기를 감지하기 위한 버클 전자 장치를 포함하는 ...,20190716,NaN,kr20190085948a,1020190085948,1020200053397,NaN,NaN,NaN
23048,2019,앞 쳐짐을 방지하는 작업용 조끼,A,20200602,"본 발명은 작업용조끼에 관한 것으로, 특히 작업용조끼의 전면에 공구 등의 중량체를 ...",A41D,A41D-001/04,2020,A41D-001/04||A41D-027/20,A41,개폐수단에 의하여 착용이 가능하게 개폐되어지며 다양한 도구 등의 수납을 위한 복수의...,20190712,102120590,kr20190084129b1,1020190084129,NaN,NaN,NaN,NaN


In [38]:
raw_df.shape

(23050, 19)

In [39]:
directory_path = '/content/라벨링데이터'
label_file_list = list_all_files_in_directory(directory_path)
label_file_list

['/content/라벨링데이터/A_농업_임업및어업_01_03/02_임업/020_임업/02012_육림업.json',
 '/content/라벨링데이터/A_농업_임업및어업_01_03/02_임업/020_임업/02030_임산물채취업.json',
 '/content/라벨링데이터/A_농업_임업및어업_01_03/02_임업/020_임업/02040_임업관련서비스업.json',
 '/content/라벨링데이터/A_농업_임업및어업_01_03/02_임업/020_임업/02011_임업용종묘생산업.json',
 '/content/라벨링데이터/A_농업_임업및어업_01_03/03_어업/031_어로어업/03111_원양어업.json',
 '/content/라벨링데이터/A_농업_임업및어업_01_03/03_어업/031_어로어업/03112_연근해어업.json',
 '/content/라벨링데이터/A_농업_임업및어업_01_03/03_어업/032_양식어업및어업관련서비스업/03211_해수면양식어업.json',
 '/content/라벨링데이터/A_농업_임업및어업_01_03/03_어업/032_양식어업및어업관련서비스업/03212_내수면양식어업.json',
 '/content/라벨링데이터/A_농업_임업및어업_01_03/03_어업/032_양식어업및어업관련서비스업/03220_어업관련서비스업.json',
 '/content/라벨링데이터/A_농업_임업및어업_01_03/03_어업/032_양식어업및어업관련서비스업/03213_수산물부화및수산종자생산업.json',
 '/content/라벨링데이터/A_농업_임업및어업_01_03/01_농업/012_축산업/01239_기타가금류및조류사육업.json',
 '/content/라벨링데이터/A_농업_임업및어업_01_03/01_농업/012_축산업/01231_양계업.json',
 '/content/라벨링데이터/A_농업_임업및어업_01_03/01_농업/012_축산업/01299_그외기타축산업.json',
 '/content/라벨링데이터/A_농업_임업및어업_01_03/01_농업/012_축산업/0121

In [40]:
len(label_file_list)

461

In [41]:
dataframes = []
for idx, file in enumerate(label_file_list):
    print(idx, file)
    data = load_json_dataset(file)
    df = pd.DataFrame(data)
    dataframes.append(df)  # 리스트에 DataFrame 추가

label_df = pd.concat(dataframes, ignore_index=True)

0 /content/라벨링데이터/A_농업_임업및어업_01_03/02_임업/020_임업/02012_육림업.json
1 /content/라벨링데이터/A_농업_임업및어업_01_03/02_임업/020_임업/02030_임산물채취업.json
2 /content/라벨링데이터/A_농업_임업및어업_01_03/02_임업/020_임업/02040_임업관련서비스업.json
3 /content/라벨링데이터/A_농업_임업및어업_01_03/02_임업/020_임업/02011_임업용종묘생산업.json
4 /content/라벨링데이터/A_농업_임업및어업_01_03/03_어업/031_어로어업/03111_원양어업.json
5 /content/라벨링데이터/A_농업_임업및어업_01_03/03_어업/031_어로어업/03112_연근해어업.json
6 /content/라벨링데이터/A_농업_임업및어업_01_03/03_어업/032_양식어업및어업관련서비스업/03211_해수면양식어업.json
7 /content/라벨링데이터/A_농업_임업및어업_01_03/03_어업/032_양식어업및어업관련서비스업/03212_내수면양식어업.json
8 /content/라벨링데이터/A_농업_임업및어업_01_03/03_어업/032_양식어업및어업관련서비스업/03220_어업관련서비스업.json
9 /content/라벨링데이터/A_농업_임업및어업_01_03/03_어업/032_양식어업및어업관련서비스업/03213_수산물부화및수산종자생산업.json
10 /content/라벨링데이터/A_농업_임업및어업_01_03/01_농업/012_축산업/01239_기타가금류및조류사육업.json
11 /content/라벨링데이터/A_농업_임업및어업_01_03/01_농업/012_축산업/01231_양계업.json
12 /content/라벨링데이터/A_농업_임업및어업_01_03/01_농업/012_축산업/01299_그외기타축산업.json
13 /content/라벨링데이터/A_농업_임업및어업_01_03/01_농업/012_축산업/01211_젖소사육업.json
14 /conte

In [42]:
label_df

,updateDate,LLno,SSno,Stext,SStext,Mtext,Lno,ipc_main,Mno,country_code,Sno,LLtext,documentId,application_number,Ltext,keyword,document_type
0,20220608,A,02012,영림업,육림업,임업,02,A01G-017/14,020,KR,0201,"농업, 임업 및 어업(01~03)",kr20210033985b1,1020210033985,임업,산림관리/임야수목관리/수목보호지지대/버팀부재/밀착부재/고정위치를용이하게조정/장치/중...,B1
1,20220608,A,02012,영림업,육림업,임업,02,A01G-017/14,020,KR,0201,"농업, 임업 및 어업(01~03)",kr20210000760u,2020210000760,임업,산림관리/산림조림/나무보호장치/땅속과나무에설치/미관상문제해결/사고예방/장치/기초보호...,U
2,20220608,A,02012,영림업,육림업,임업,02,A01G-017/14,020,KR,0201,"농업, 임업 및 어업(01~03)",kr20210006694b1,1020210006694,임업,산림관리/임야수목관리/수목관리장치/안정적지지/유수공급/수목효율적관리/장치/입수부/조...,B1
3,20220608,A,02012,영림업,육림업,임업,02,A01G-023/04,020,KR,0201,"농업, 임업 및 어업(01~03)",kr20210001336b1,1020210001336,임업,산림관리/산림조림/안정지지대/지지바아/각도구속부/휴대용안정지지대/장치/제1지지바아/...,B1
4,20220608,A,02012,영림업,육림업,임업,02,A01G-013/02,020,KR,0201,"농업, 임업 및 어업(01~03)",kr20200152106b1,1020200152106,임업,산림관리/임야수목관리/블록/천연골재로제작/식재공간변형/물고임방지/장치/결합홈/결합돌...,B1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23045,20220608,C,14499,기타 의복 액세서리 제조업,그 외 기타 의복 액세서리 제조업,의복 액세서리 제조업,14,A41F-009/00,144,KR,1449,제조업(10~34),kr20207036302a,1020207036302,"의복, 의복 액세서리 및 모피제품 제조업",벨트제조//벨트/피복/고정/억제/조성물/버클본체/계지부/계합부,A
23046,20220608,C,14499,기타 의복 액세서리 제조업,그 외 기타 의복 액세서리 제조업,의복 액세서리 제조업,14,A41F-009/02,144,KR,1449,제조업(10~34),kr20190092477a,1020190092477,"의복, 의복 액세서리 및 모피제품 제조업",벨트제조//벨트/고정부/긴띠/고정/조성물/조절/고리/스프링클립,A
23047,20220608,C,14499,기타 의복 액세서리 제조업,그 외 기타 의복 액세서리 제조업,의복 액세서리 제조업,14,A41F-009/00,144,KR,1449,제조업(10~34),kr20190085948a,1020190085948,"의복, 의복 액세서리 및 모피제품 제조업",벨트제조//스마트벨트/접촉부/압력/자세교정/장치/버클/밴드/경고신호,A
23048,20220608,C,14499,기타 의복 액세서리 제조업,그 외 기타 의복 액세서리 제조업,의복 액세서리 제조업,14,A41D-001/04,144,KR,1449,제조업(10~34),kr20190084129b1,1020190084129,"의복, 의복 액세서리 및 모피제품 제조업",의류액세서리제조//조끼/견갑골/무게하중/착용감/장치/등판부/개구홈/측면오목부,B1


In [43]:
label_df.shape

(23050, 17)

In [44]:
특허_카테고리_list = label_df['Ltext'].unique().tolist()
특허_카테고리_list

['임업',
 '어업',
 '농업',
 '석탄, 원유 및 천연가스 광업',
 '광업 지원 서비스업',
 '금속 광업',
 '비금속광물 광업; 연료용 제외',
 '산업용 기계 및 장비 수리업',
 '화학 물질 및 화학제품 제조업; 의약품 제외',
 '전자 부품, 컴퓨터, 영상, 음향 및 통신장비 제조업',
 '코크스, 연탄 및 석유정제품 제조업',
 '고무 및 플라스틱제품 제조업',
 '자동차 및 트레일러 제조업',
 '기타 운송장비 제조업',
 '목재 및 나무제품 제조업; 가구 제외',
 '펄프, 종이 및 종이제품 제조업',
 '음료 제조업',
 '비금속 광물제품 제조업',
 '인쇄 및 기록매체 복제업',
 '금속 가공제품 제조업; 기계 및 가구 제외',
 '의료용 물질 및 의약품 제조업',
 '의료, 정밀, 광학 기기 및 시계 제조업',
 '기타 제품 제조업',
 '담배 제조업',
 '기타 기계 및 장비 제조업',
 '가구 제조업',
 '식료품 제조업',
 '1차 금속 제조업',
 '전기장비 제조업',
 '섬유제품 제조업; 의복 제외',
 '가죽, 가방 및 신발 제조업',
 '의복, 의복 액세서리 및 모피제품 제조업']

In [45]:
len(특허_카테고리_list)

32

# 중복된 값 제거

In [46]:
# raw_df에서 중복된 application_number 확인
print(raw_df['application_number'].duplicated().sum())

# label_df에서 중복된 application_number 확인
print(label_df['application_number'].duplicated().sum())

3538
3538


In [47]:
# 중복된 application_number 찾기
duplicate_application_numbers = raw_df[raw_df['application_number'].duplicated()]['application_number'].unique()
duplicate_application_numbers

array(['1020210033985', '1020200152106', '1020200088844', ...,
       '1020190109607', '1020190106033', '2020190002754'], dtype=object)

In [48]:
duplicate_rows = raw_df[raw_df['application_number'].isin(['1020190179476'])]

In [49]:
duplicate_rows

,application_year,invention_title,ipc_section,register_date,abstract,ipc_subclass,ipc_main,register_year,ipc_all,ipc_class,claims,application_date,register_number,documentId,application_number,open_number,open_date,open_year,applicant_name
1617,2019,회처리 설비용 철편 분리 장치,B,NaN,"본 발명은 회처리 설비용 철편 분리 장치에 관한 것으로, 석탄회(비회, 저회)의 이...",B03C,B03C-001/28,NaN,B03C-001/28||B03C-001/30||F16B-005/02||G08B-00...,B03,석탄회를 이송하는 관로에 외부가 통하도록 개방되는 철편 분리공간을 밀폐하며 상기 관...,20191231,NaN,kr20190179476a,1020190179476,1020210085907,NaN,NaN,NaN
1618,2019,회처리 설비용 철편 분리 장치,B,20211014,"본 발명은 회처리 설비용 철편 분리 장치에 관한 것으로, 석탄회(비회, 저회)의 이...",B03C,B03C-001/28,2021,B03C-001/28||B03C-001/30||F16B-005/02||G08B-00...,B03,석탄회를 이송하는 관로에 외부가 통하도록 개방되는 철편 분리공간을 밀폐하며 상기 관...,20191231,102315240,kr20190179476b1,1020190179476,NaN,NaN,NaN,NaN
1814,2019,회처리 설비용 철편 분리 장치,B,NaN,"본 발명은 회처리 설비용 철편 분리 장치에 관한 것으로, 석탄회(비회, 저회)의 이...",B03C,B03C-001/28,NaN,B03C-001/28||B03C-001/30||F16B-005/02||G08B-00...,B03,석탄회를 이송하는 관로에 외부가 통하도록 개방되는 철편 분리공간을 밀폐하며 상기 관...,20191231,NaN,kr20190179476a,1020190179476,1020210085907,NaN,NaN,NaN


In [50]:
# raw_df에서 application_number 기준으로 중복된 행 제거
raw_df_unique = raw_df.drop_duplicates(subset='application_number', keep='first')

# label_df에서 application_number 기준으로 중복된 행 제거
label_df_unique = label_df.drop_duplicates(subset='application_number', keep='first')

In [51]:
raw_df_unique.shape

(19512, 19)

In [52]:
label_df_unique.shape

(19512, 17)

In [53]:
merged_df = pd.merge(raw_df_unique, label_df_unique, on='application_number', how='inner')
merged_df

,application_year,invention_title,ipc_section,register_date,abstract,ipc_subclass,ipc_main_x,register_year,ipc_all,ipc_class,...,Lno,ipc_main_y,Mno,country_code,Sno,LLtext,documentId_y,Ltext,keyword,document_type
0,2021,수목 보호 지지대,A,20210609,"본 발명은 수목 보호 지지대에 관한 것으로, 지면에 하단이 고정되는 다수의 버팀부재...",A01G,A01G-017/14,2021,A01G-017/14||A01G-017/10||A01G-009/12,A01,...,02,A01G-017/14,020,KR,0201,"농업, 임업 및 어업(01~03)",kr20210033985b1,임업,산림관리/임야수목관리/수목보호지지대/버팀부재/밀착부재/고정위치를용이하게조정/장치/중...,B1
1,2021,나무보호장치,A,NaN,강풍에 나무들이나 전봇대들이 뽑히거나 부러지고 넘어져서 피해를 입는 사고가 많다. ...,A01G,A01G-017/14,NaN,A01G-017/14||A01G-009/12,A01,...,02,A01G-017/14,020,KR,0201,"농업, 임업 및 어업(01~03)",kr20210000760u,임업,산림관리/산림조림/나무보호장치/땅속과나무에설치/미관상문제해결/사고예방/장치/기초보호...,U
2,2021,수목 관리 장치 및 시스템,A,20210423,"본 발명은 수목 관리 장치 및 시스템에 관한 것으로, 더욱 상세하게는 수목을 안정적...",A01G,A01G-017/14,2021,A01G-017/14||A01G-027/00||F21S-008/08||F21S-00...,A01,...,02,A01G-017/14,020,KR,0201,"농업, 임업 및 어업(01~03)",kr20210006694b1,임업,산림관리/임야수목관리/수목관리장치/안정적지지/유수공급/수목효율적관리/장치/입수부/조...,B1
3,2021,휴대용 나무 안정지지대,A,20210601,"본 발명은 휴대용 나무 안정지지대에 관한 것으로서, 회동축(A)을 중심으로 크로스 ...",A01G,A01G-023/04,2021,A01G-023/04||A01G-017/14,A01,...,02,A01G-023/04,020,KR,0201,"농업, 임업 및 어업(01~03)",kr20210001336b1,임업,산림관리/산림조림/안정지지대/지지바아/각도구속부/휴대용안정지지대/장치/제1지지바아/...,B1
4,2020,수목보호용 블록,A,20210611,본 발명은 블록의 조립에 의해 이루어진 식재 공간 내부에 식재되는 수목의 위치에 맞...,A01G,A01G-013/02,2021,A01G-013/02||C04B-020/00||C04B-028/02||E01C-00...,A01,...,02,A01G-013/02,020,KR,0201,"농업, 임업 및 어업(01~03)",kr20200152106b1,임업,산림관리/임야수목관리/블록/천연골재로제작/식재공간변형/물고임방지/장치/결합홈/결합돌...,B1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19461,2019,운동용 파워 벨트부재의 제조방법 및 이를 이용한 운동용 허리벨트,A,NaN,본 발명은 운동용 파워 벨트부재의 제조방법 및 이를 이용한 운동용 허리벨트에 관한 ...,A41F,A41F-009/00,NaN,A41F-009/00||A41D-013/05||A41D-027/06,A41,...,14,A41F-009/00,144,KR,1449,제조업(10~34),kr20190112005a,"의복, 의복 액세서리 및 모피제품 제조업",벨트제조//벨트/메쉬/압박대/운동/방법/원단/중첩부/재봉부,A
19462,2019,"버클, 벨트 및 벨트 부품",A,NaN,"피복의 흘러내림을 억제하는 벨트 버클로서, 벨트의 일단이 고정된 버클 본체와, 버클...",A41F,A41F-009/00,NaN,A41F-009/00||A41F-017/02||A44B-011/00||A44B-01...,A41,...,14,A41F-009/00,144,KR,1449,제조업(10~34),kr20207036302a,"의복, 의복 액세서리 및 모피제품 제조업",벨트제조//벨트/피복/고정/억제/조성물/버클본체/계지부/계합부,A
19463,2019,히든 벨트,A,NaN,"본 발명은 히든벨트에 대한 것으로서, 본 발명의 일실시예에 따른 벨트는 고정부와 긴...",A41F,A41F-009/02,NaN,A41F-009/02||A41F-009/00,A41,...,14,A41F-009/02,144,KR,1449,제조업(10~34),kr20190092477a,"의복, 의복 액세서리 및 모피제품 제조업",벨트제조//벨트/고정부/긴띠/고정/조성물/조절/고리/스프링클립,A
19464,2019,버클 전자 장치를 포함하는 스마트 벨트,A,NaN,"본 발명의 실시 예에 따른 스마트 벨트는, 착용 부위의 복부 압력 및 자세 기울기를...",A41F,A41F-009/00,NaN,A41F-009/00||A44B-011/00||A61B-005/00||A61B-00...,A41,...,14,A41F-009/00,144,KR,1449,제조업(10~34),kr20190085948a,"의복, 의복 액세서리 및 모피제품 제조업",벨트제조//스마트벨트/접촉부/압력/자세교정/장치/버클/밴드/경고신호,A


In [54]:
merged_df.shape

(19466, 35)

In [55]:
def extract_fields(row):
    # 각 행(row)에서 필요한 필드를 추출
    invention_title = row.get('invention_title', '')
    abstract = row.get('abstract', '')
    claims = row.get('claims', '')

    # 추출한 내용을 하나의 문자열로 연결
    result = f"invention_title: {invention_title} abstract: {abstract} claims: {claims}"

    return result

# apply 함수를 사용하여 merged_df의 각 행에 대해 extract_fields 함수를 적용
merged_df['combined_string'] = merged_df.apply(extract_fields, axis=1)

# 결과 출력 (예시로 첫 5개의 결과 출력)
merged_df[['invention_title', 'abstract', 'claims', 'combined_string']].head()

,invention_title,abstract,claims,combined_string
0,수목 보호 지지대,"본 발명은 수목 보호 지지대에 관한 것으로, 지면에 하단이 고정되는 다수의 버팀부재...","수목(100)의 둘레 면에 접촉되는 다수의 밀착부재(2)와, 상기 밀착부재(2)를 ...",invention_title: 수목 보호 지지대 abstract: 본 발명은 수목 ...
1,나무보호장치,강풍에 나무들이나 전봇대들이 뽑히거나 부러지고 넘어져서 피해를 입는 사고가 많다. ...,'기초보호대' 와 '땅속지지대' 를 땅속에 설치하여 강풍에 나무가 옆으로 기울거나 ...,invention_title: 나무보호장치 abstract: 강풍에 나무들이나 전봇...
2,수목 관리 장치 및 시스템,"본 발명은 수목 관리 장치 및 시스템에 관한 것으로, 더욱 상세하게는 수목을 안정적...","수목(1)을 감싸도록 주변에 설치되어 수목(1)을 지지하고, 유수를 집수하여 수목(...",invention_title: 수목 관리 장치 및 시스템 abstract: 본 발명...
3,휴대용 나무 안정지지대,"본 발명은 휴대용 나무 안정지지대에 관한 것으로서, 회동축(A)을 중심으로 크로스 ...","회동축(A)을 중심으로 크로스 결합된 것으로서, 땅(E)에 지지되어 이식 대기중인 ...",invention_title: 휴대용 나무 안정지지대 abstract: 본 발명은 ...
4,수목보호용 블록,본 발명은 블록의 조립에 의해 이루어진 식재 공간 내부에 식재되는 수목의 위치에 맞...,수목의 주위에 설치되어 수목이 설치된 지반의 흙이 외부로 흩어지는 것을 방지하는 수...,invention_title: 수목보호용 블록 abstract: 본 발명은 블록의 ...


In [56]:
merged_df

,application_year,invention_title,ipc_section,register_date,abstract,ipc_subclass,ipc_main_x,register_year,ipc_all,ipc_class,...,ipc_main_y,Mno,country_code,Sno,LLtext,documentId_y,Ltext,keyword,document_type,combined_string
0,2021,수목 보호 지지대,A,20210609,"본 발명은 수목 보호 지지대에 관한 것으로, 지면에 하단이 고정되는 다수의 버팀부재...",A01G,A01G-017/14,2021,A01G-017/14||A01G-017/10||A01G-009/12,A01,...,A01G-017/14,020,KR,0201,"농업, 임업 및 어업(01~03)",kr20210033985b1,임업,산림관리/임야수목관리/수목보호지지대/버팀부재/밀착부재/고정위치를용이하게조정/장치/중...,B1,invention_title: 수목 보호 지지대 abstract: 본 발명은 수목 ...
1,2021,나무보호장치,A,NaN,강풍에 나무들이나 전봇대들이 뽑히거나 부러지고 넘어져서 피해를 입는 사고가 많다. ...,A01G,A01G-017/14,NaN,A01G-017/14||A01G-009/12,A01,...,A01G-017/14,020,KR,0201,"농업, 임업 및 어업(01~03)",kr20210000760u,임업,산림관리/산림조림/나무보호장치/땅속과나무에설치/미관상문제해결/사고예방/장치/기초보호...,U,invention_title: 나무보호장치 abstract: 강풍에 나무들이나 전봇...
2,2021,수목 관리 장치 및 시스템,A,20210423,"본 발명은 수목 관리 장치 및 시스템에 관한 것으로, 더욱 상세하게는 수목을 안정적...",A01G,A01G-017/14,2021,A01G-017/14||A01G-027/00||F21S-008/08||F21S-00...,A01,...,A01G-017/14,020,KR,0201,"농업, 임업 및 어업(01~03)",kr20210006694b1,임업,산림관리/임야수목관리/수목관리장치/안정적지지/유수공급/수목효율적관리/장치/입수부/조...,B1,invention_title: 수목 관리 장치 및 시스템 abstract: 본 발명...
3,2021,휴대용 나무 안정지지대,A,20210601,"본 발명은 휴대용 나무 안정지지대에 관한 것으로서, 회동축(A)을 중심으로 크로스 ...",A01G,A01G-023/04,2021,A01G-023/04||A01G-017/14,A01,...,A01G-023/04,020,KR,0201,"농업, 임업 및 어업(01~03)",kr20210001336b1,임업,산림관리/산림조림/안정지지대/지지바아/각도구속부/휴대용안정지지대/장치/제1지지바아/...,B1,invention_title: 휴대용 나무 안정지지대 abstract: 본 발명은 ...
4,2020,수목보호용 블록,A,20210611,본 발명은 블록의 조립에 의해 이루어진 식재 공간 내부에 식재되는 수목의 위치에 맞...,A01G,A01G-013/02,2021,A01G-013/02||C04B-020/00||C04B-028/02||E01C-00...,A01,...,A01G-013/02,020,KR,0201,"농업, 임업 및 어업(01~03)",kr20200152106b1,임업,산림관리/임야수목관리/블록/천연골재로제작/식재공간변형/물고임방지/장치/결합홈/결합돌...,B1,invention_title: 수목보호용 블록 abstract: 본 발명은 블록의 ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19461,2019,운동용 파워 벨트부재의 제조방법 및 이를 이용한 운동용 허리벨트,A,NaN,본 발명은 운동용 파워 벨트부재의 제조방법 및 이를 이용한 운동용 허리벨트에 관한 ...,A41F,A41F-009/00,NaN,A41F-009/00||A41D-013/05||A41D-027/06,A41,...,A41F-009/00,144,KR,1449,제조업(10~34),kr20190112005a,"의복, 의복 액세서리 및 모피제품 제조업",벨트제조//벨트/메쉬/압박대/운동/방법/원단/중첩부/재봉부,A,invention_title: 운동용 파워 벨트부재의 제조방법 및 이를 이용한 운동...
19462,2019,"버클, 벨트 및 벨트 부품",A,NaN,"피복의 흘러내림을 억제하는 벨트 버클로서, 벨트의 일단이 고정된 버클 본체와, 버클...",A41F,A41F-009/00,NaN,A41F-009/00||A41F-017/02||A44B-011/00||A44B-01...,A41,...,A41F-009/00,144,KR,1449,제조업(10~34),kr20207036302a,"의복, 의복 액세서리 및 모피제품 제조업",벨트제조//벨트/피복/고정/억제/조성물/버클본체/계지부/계합부,A,"invention_title: 버클, 벨트 및 벨트 부품 abstract: 피복의 ..."
19463,2019,히든 벨트,A,NaN,"본 발명은 히든벨트에 대한 것으로서, 본 발명의 일실시예에 따른 벨트는 고정부와 긴...",A41F,A41F-009/02,NaN,A41F-009/02||A41F-009/00,A41,...,A41F-009/02,144,KR,1449,제조업(10~34),kr20190092477a,"의복, 의복 액세서리 및 모피제품 제조업",벨트제조//벨트/고정부/긴띠/고정/조성물/조절/고리/스프링클립,A,invention_title: 히든 벨트 abstract: 본 발명은 히든벨트에 대...
19464,2019,버클 전자 장치를 포함하는 스마트 벨트,A,NaN,"본 발명의 실시 예에 따른 스마트 벨트는, 착용 부위의 복부 압력 및 자세 기울기를...",A41F,A41F-009/00,NaN,A41F-009/00||A44B-011/00||A61B-005/00||A61B-00...,A41,...,A41F-009/00,144,KR,1449,제조업(10~34),kr20190085948a,"의복, 의복 액세서리 및 모피제품 제조업",벨트제조//스마트벨트/접촉부/압력/자세교정/장치/버클/밴드/경고신호,A,invention_title: 버클 전자 장치를 포함하는 스마트 벨트 abstrac...


In [57]:
test_data_df = merged_df[['application_number', 'combined_string', 'Ltext']]
test_data_df

,application_number,combined_string,Ltext
0,1020210033985,invention_title: 수목 보호 지지대 abstract: 본 발명은 수목 ...,임업
1,2020210000760,invention_title: 나무보호장치 abstract: 강풍에 나무들이나 전봇...,임업
2,1020210006694,invention_title: 수목 관리 장치 및 시스템 abstract: 본 발명...,임업
3,1020210001336,invention_title: 휴대용 나무 안정지지대 abstract: 본 발명은 ...,임업
4,1020200152106,invention_title: 수목보호용 블록 abstract: 본 발명은 블록의 ...,임업
...,...,...,...
19461,1020190112005,invention_title: 운동용 파워 벨트부재의 제조방법 및 이를 이용한 운동...,"의복, 의복 액세서리 및 모피제품 제조업"
19462,1020207036302,"invention_title: 버클, 벨트 및 벨트 부품 abstract: 피복의 ...","의복, 의복 액세서리 및 모피제품 제조업"
19463,1020190092477,invention_title: 히든 벨트 abstract: 본 발명은 히든벨트에 대...,"의복, 의복 액세서리 및 모피제품 제조업"
19464,1020190085948,invention_title: 버클 전자 장치를 포함하는 스마트 벨트 abstrac...,"의복, 의복 액세서리 및 모피제품 제조업"


In [58]:
# '농업', '임업', '어업' 데이터만 추출
filtered_df = test_data_df[test_data_df['Ltext'].isin(['농업', '임업', '어업'])]
filtered_df

,application_number,combined_string,Ltext
0,1020210033985,invention_title: 수목 보호 지지대 abstract: 본 발명은 수목 ...,임업
1,2020210000760,invention_title: 나무보호장치 abstract: 강풍에 나무들이나 전봇...,임업
2,1020210006694,invention_title: 수목 관리 장치 및 시스템 abstract: 본 발명...,임업
3,1020210001336,invention_title: 휴대용 나무 안정지지대 abstract: 본 발명은 ...,임업
4,1020200152106,invention_title: 수목보호용 블록 abstract: 본 발명은 블록의 ...,임업
...,...,...,...
1145,1020200133095,invention_title: 수경재배기 abstract: 본 발명은 수경재배기에 ...,농업
1146,1020200133158,invention_title: 분무식 수경 재배장치 abstract: 본 발명은 본...,농업
1147,1020200133159,invention_title: 재배용 포트 abstract: 본 발명은 상하가 개방...,농업
1148,1020200121174,invention_title: 거치용 식물 재배 장치 abstract: 본 발명에 ...,농업


In [59]:
import csv

# 1. 기존 CSV 파일을 읽어서 application_number 목록을 추출
try:
    existing_df = pd.read_csv('test_output.csv', encoding='utf-8')
    existing_app_numbers = set(existing_df['application_number'].astype(str))
except FileNotFoundError:
    # 파일이 없을 경우 빈 set으로 초기화
    existing_app_numbers = set()

In [ ]:
total_num = len(filtered_df)

# 2. 새 데이터를 추가할 CSV 파일을 연다
with open('test_output.csv', mode='a', newline='', encoding='utf-8') as file:
    writer = csv.writer(file, quoting=csv.QUOTE_MINIMAL)

    # 3. 데이터프레임 순회하면서 application_number가 중복되지 않으면 추가
    for i, (index, row) in enumerate(filtered_df.iterrows()):
        app_number = str(row['application_number'])  # 문자열로 변환하여 비교
        if app_number not in existing_app_numbers:
            llama3_1_inference_result = generate_response(system_message=system_prompt,
                                          user_message=row['combined_string'],
                                          tokenizer=tokenizer,
                                          model=model)

            writer.writerow([i, row['application_number'], row['combined_string'], row['Ltext'], llama3_1_inference_result])
            print(f"{i}/{total_num} Row {i}: application_number: {row['application_number']}, combined_string: {row['combined_string']}, Ltext: {row['Ltext']}, prediction: {llama3_1_inference_result}")
            existing_app_numbers.add(app_number)  # 새로 추가된 번호는 추적
        else:
            print(f"Skipping Row {i}: application_number {row['application_number']} already exists.")

/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention

0/1059 Row 0: application_number: 1020210033985, combined_string: invention_title: 수목 보호 지지대 abstract: 본 발명은 수목 보호 지지대에 관한 것으로, 지면에 하단이 고정되는 다수의 버팀부재의 상단을 용이하게 고정시키는 본체 및 수목의 둘레 면에 접하여 지지하는 다수의 밀착부재 그리고 상기 버팀부재 및 밀착부재를 안정적으로 본체에 장착될 수 있도록 개량한 구조가 간결하게 이루어짐으로써 생산성 향상과 간편하게 설치할 수 있는 작업의 효율성을 높일 수 있고, 안정적으로 수목이 성장되도록 지지함은 물론 수목의 성장에 따른 굵기(체적)의 변화에 따른 밀착부재의 고정위치를 용이하게 조정할 수 있도록 이루어지는 수목 보호 지지대에 관한 것이다. claims: 수목(100)의 둘레 면에 접촉되는 다수의 밀착부재(2)와, 상기 밀착부재(2)를 안착시키면서 수목(100)을 중앙에 위치시키도록 일측이 개방된 중공부(11)를 갖는 본체(1)와, 상기 본체(1)의 개방부분을 폐쇄하는 고정편(3) 및 상기 본체(1)에 상단이 고정되고 하단이 지면에 고정되는 다수의 버팀부재(4)로 이루어진 수목 보호 지지대에 있어서,상기 본체(1)는 수목(100)이 위치하는 중공부(11)를 기준으로 3~4개소에 상기 버팀부재(4)의 상단이 돌출되면서 끼워지도록 천공된 다수의 끼움구멍(12)과, 이 끼움구멍(12)과 끼움구멍(12) 사이에 형성되어 상기 밀착부재(2)를 고정하는 고정볼트가 체결되는 나사공(13) 및 이탈을 방지하면서 수목(100)의 중심으로 직선 이동할 수 있도록 밀착부재(2)를 안내하는 가이드편(14)이 돌출 형성되어 이루어지고;상기 고정편(3)은 고정볼트에 의해 장착되어 본체(1)의 개방부분을 폐쇄하면서 내구성을 증대시키고 상기 밀착부재(2)를 고정하는 고정볼트가 체결되는 나사공(31) 및 이탈을 방지하면서 수목(100)의 중심으로 직선 이동할 수 있도록 밀착부재(2)를 안내하는 가이드편(32)이 돌출 형성

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


2/1059 Row 2: application_number: 1020210012672, combined_string: invention_title: 컨테이너 조경수 생산 및 유통 관리를 위한 ERP 시스템과 그 방법 abstract: 본 발명은 컨테이너 조경수 생산 및 유통 관리를 위한 ERP 시스템과 그 방법을 개시한다. 즉, 본 발명은 컨테이너에서 재배되는 수목인 컨테이너 조경수의 생육 데이터, 환경 데이터 등을 근거로 딥러닝 또는 기계 학습을 통해 생육 데이터를 예측하고, 생산성 환경을 분석하여, 최적의 조경수 생산을 위한 생산성 환경을 추천함으로써, 데이터 기반의 표준화된 조경수 데이터베이스를 구축하고, 조경수 생산을 위한 스마트 재배 시스템의 전체 운영 효율을 향상시킬 수 있다. claims: 앱 실행 결과 화면 중에서 미리 설정된 생육 예측 데이터 메뉴가 선택될 때, 조경수에 대한 생육과 관련한 예측 데이터를 확인하기 위한 생육 예측 데이터 화면을 표시하고, 썸네일 형태로 표시되는 복수의 조경수 중에서 어느 하나의 특정 조경수가 선택될 때, 상기 선택된 특정 조경수에 대응하는 생육 예측 데이터를 표시하는 단말;상기 단말의 요청에 따라 복수의 조경수의 고유 식별 정보별 생육 예측 데이터 중에서 상기 조경수의 고유 식별 정보에 대응하는 생육 예측 데이터를 확인하고, 상기 확인된 상기 조경수의 고유 식별 정보에 대응하는 생육 예측 데이터를 상기 단말에 제공하는 ERP 서버; 특정 조경수와 관련한 영상 정보, 생육 데이터 및 환경 데이터를 수집하는 데이터 수집부; 및복수의 조경수로 구성된 농장의 환경을 제어하는 스마트 재배 시스템을 포함하며,상기 ERP 서버는,상기 데이터 수집부로부터 전송되는 특정 조경수와 관련한 영상 정보, 생육 데이터 및 환경 데이터를 수신하고, 상기 영상 정보에 포함된 특정 조경수를 미리 설정된 수형 이미지 분류 모델의 입력값으로 하여 기계 학습을 수행하고, 기계 학습 결과를 근거로 상기 영상 정보에 포함된 특정 조경수의 수형, 높이

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


...

1052/1059 Row 1052: application_number: 1020190118397, combined_string: invention_title: 구기자를 이용한 고품질 가축류의 사육방법 abstract: 본 발명은 구기자를 이용한 고품질 가축류의 사육방법으로(a) 구기자 : 구기자잎을 10~40 : 60~90의 중량비로 혼합하는 단계; (b) 상기(a)의 구기자혼합물을 100㎛ 내지 1000㎛의 크기로 분쇄하여 준비하는 단계; (c) 식용버섯을 20㎛ 내지 50㎛의 크기로 분쇄하여 준비하는 단계; (d) 상기(b)단계의 구기자혼합물 : 상기(c)단계의 버섯분말을 20~60 : 40~80의 중량비로 혼합하여 준비하는 단계; (e) 상기(d)단계의 혼합물에 발효효소를 접종하는 단계; (f) 상기(e)단계의 발효효소를 접종한 구기자혼합물을 20~35℃의 온도에서 3일~5일간 발효하는 단계; (g) 상기(f)단계의 발효가 완료된 구기자혼합물을 건조하는 단계; (h) 일반 가축류 급여용 사료 1000kg당 상기(g)단계의 건조된 구기자혼합물을 1kg~10kg을 혼합하여 교반하는 단계; (i) 상기(h)단계의 혼합사료를 가축류 출하전 60일~180일간 급여하여 사육하는 단계; (j) 상기(i)단계의 사육과정을 통하여 사육된 가축류를 출하하여 제품화하는 단계를 포함하여 이루어진다. claims: (a) 구기자 : 구기자잎을 10~40 : 60~90의 중량비로 혼합하는 단계; (b) 상기(a)의 구기자혼합물을 100㎛ 내지 1000㎛의 크기로 분쇄하여 준비하는 단계; (c) 식용버섯을 20㎛ 내지 50㎛의 크기로 분쇄하여 준비하는 단계; (d) 상기(b)단계의 구기자혼합물 : 상기(c)단계의 버섯분말을 20~60 : 40~80의 중량비로 혼합하여 준비하는 단계; (e) 상기(d)단계의 혼합물에 발효효소를 접종하는 단계; (f) 상기(e)단계의 발효효소를 접종한 구기자혼합물을 20~35℃의 온도에서 3일~5일간 발효하는 단계; (g) 상기(f)단계의 발효가 완료

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


1054/1059 Row 1054: application_number: 1020190115691, combined_string: invention_title: 그룹별 축산물 생애 정보 제공 시스템 abstract: 본 발명은 그룹별 축산물 생애 정보 제공 시스템에 관한 것으로서, 보다 구체적으로는 정보 제공 시스템으로서, 정보 제공 시스템으로서, 가축 및 가금을 포함하는 축산물에 대한 정보를 제공하되, 서로 관계있는 복수의 축산물을 그룹화하여 그룹별로 축산물의 건강관리 정보, 사육 환경 정보 및 농장 정보를 포함하는 축산물 생애 정보를 관리 및 제공하는 정보 제공 서버; 및 상기 축산물의 식별정보를 인식하고, 인식한 식별정보를 상기 정보 제공 서버에 전송하여 상기 축산물에 대한 데이터 제공 요청을 하는 소비자 디바이스를 포함하며, 상기 소비자 디바이스는, 상기 축산물의 식별정보를 인식하는 인식 모듈; 상기 정보 제공 서버에 상기 식별정보를 전달하고, 상기 식별정보에 대응되는 축산물의 데이터 제공 요청을 하는 정보 요청 모듈; 및 상기 정보 제공 서버로부터 수신한 축산물 생애 정보를 출력하는 출력 모듈을 포함하는 것을 그 구성상의 특징으로 한다.가축 및 가금을 포함하는 축산물에 대한 정보를 제공하는 정보 제공 서버; 및 상기 축산물의 식별정보를 인식하고, 인식한 식별정보를 상기 정보 제공 서버에 전송하여 상기 축산물에 대한 데이터 제공 요청을 하는 소비자 디바이스를 포함하며, 상기 정보 제공 서버는, 서로 관계있는 복수의 축산물을 그룹화하고, 그룹화 한 그룹별로 식별정보를 할당하는 그룹화 모듈; 상기 그룹별로 축산물의 건강관리 정보, 사육 환경 정보 및 농장 정보를 포함하는 축산물 생애 정보를 저장하는 데이터베이스 모듈; 상기 그룹별로 축산물이 사육되는 농가의 축사 관리 기록, 축산물의 진료 또는 치료 정보, 및 측정 정보를 포함하는 건강관리 정보를 수집하는 제1 정보 수집 모듈; 상기 그룹별로 축산물이 사육되는 농가 또는 축사의 주변 환경 정보를 포함하는 사육

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


1056/1059 Row 1056: application_number: 1020190101443, combined_string: invention_title: 자동산란장치 abstract: 본 발명은 닭의 산란 환경을 조성하고 달걀 수거를 자동으로 진행할 수 있는 자동산란장치에 대한 것이며, 구체적으로 닭이 진입하여 산란하는 산란부와 산란부에 진입한 닭의 산란 환경을 조성하기 위한 산란유도부와 산란부의 하단에 형성되며, 바닥재를 수거하여 재생하는 바닥재수거부와 닭이 산란한 달걀을 수거하는 달걀수거부를 구비한다. claims: 닭이 진입하여 산란하는 산란부(100);,상기 산란부(100)에 진입한 닭의 산란 환경을 조성하기 위한 산란유도부(200);,상기 산란부(100)의 하단에 형성되며, 바닥재(20)를 수거하여 재생하는 바닥재수거부(300);닭이 산란한 달걀(10)을 수거하는 달걀수거부(400);를 포함하는 자동산란장치., Ltext: 농업, prediction: 임업
1057/1059 Row 1057: application_number: 1020190101249, combined_string: invention_title: 스마트 달걀 트레이 기반 달걀 자동주문 및 관리를 수행하기 위한 장치 및 방법 abstract: 본 발명의 일 실시예에 따라, 서버에 의해 수행되는, 스마트 달걀트레이 기반 달걀 자동주문 및 관리를 수행하기 위한 방법에 있어서, (a) 달걀 트레이로부터 부족한 달걀의 개수를 수신하는 단계; (b) 달걀의 판매가 가능한 양계장 리스트를 사용자 단말로 제공하는 단계; (c) 사용자 단말이 선택한 양계장에 대응하는 양계장의 공급자 단말로 부족한 달걀의 개수만큼 주문요청을 전송하고, 공급자 단말로부터 사용자에게 배송될 달걀 이미지를 수신하는 단계; 및 (d) 달걀 이미지를 사용자 단말로 제공하고, 공급자 단말 또는 사용자 단말을 통해 수집한 유통기한 관련정보를 참고하여, 사용자 단말로 달걀이 주문된 이후, 기 설정된 시간 경과 시 달걀 트레이 

In [ ]:
from google.colab import files

# 파일 다운로드
files.download('test_output.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# CSV 파일 읽기
df = pd.read_csv('test_output.csv', header=None)

# 값이 동일한지 확인
equal_count = (df.iloc[:, 3] == df.iloc[:, 4]).sum()

# 전체 데이터에서 동일한 값이 차지하는 비율 계산
total_rows = len(df)
equal_percentage = (equal_count / total_rows) * 100

# 결과 출력
print(f"Meta-Llama-3.1-8B-Instruct 모델 예측 결과 정확도 계산('농업','임업', '어업') (Base 버전): {equal_percentage:.2f}%")
# Meta-Llama-3.1-8B-Instruct 모델 예측 결과 정확도 계산('농업','임업', '어업') (Base 버전): 42.68%

Meta-Llama-3.1-8B-Instruct 모델 예측 결과 정확도 계산('농업','임업', '어업') (Base 버전): 43.81%


# Part 2 : 프롬프트 튜닝으로 Llama 3.1 성능 개선하기

In [60]:
# 정답이 '임업'일 때
correct_label = '임업'

# 정답이 '임업'일 때 '임업'으로 예측한 비율
correct_forest_pred_forest = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '임업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '임업'일 때 '농업'으로 예측한 비율
correct_forest_pred_agriculture = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '농업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '임업'일 때 '어업'으로 예측한 비율
correct_forest_pred_fishery = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '어업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '농업'일 때
correct_label = '농업'

# 정답이 '농업'일 때 '농업'으로 예측한 비율
correct_agriculture_pred_agriculture = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '농업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '농업'일 때 '임업'으로 예측한 비율
correct_agriculture_pred_forest = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '임업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '농업'일 때 '어업'으로 예측한 비율
correct_agriculture_pred_fishery = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '어업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '어업'일 때
correct_label = '어업'

# 정답이 '어업'일 때 '어업'으로 예측한 비율
correct_fishery_pred_fishery = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '어업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '어업'일 때 '임업'으로 예측한 비율
correct_fishery_pred_forest = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '임업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '어업'일 때 '농업'으로 예측한 비율
correct_fishery_pred_agriculture = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '농업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 결과 출력
print(f"정답이 '임업'일 때 '임업'으로 예측한 비율: {correct_forest_pred_forest:.2f}%")
print(f"정답이 '임업'일 때 '농업'으로 예측한 비율: {correct_forest_pred_agriculture:.2f}%")
print(f"정답이 '임업'일 때 '어업'으로 예측한 비율: {correct_forest_pred_fishery:.2f}%")

print(f"정답이 '농업'일 때 '농업'으로 예측한 비율: {correct_agriculture_pred_agriculture:.2f}%")
print(f"정답이 '농업'일 때 '임업'으로 예측한 비율: {correct_agriculture_pred_forest:.2f}%")
print(f"정답이 '농업'일 때 '어업'으로 예측한 비율: {correct_agriculture_pred_fishery:.2f}%")

print(f"정답이 '어업'일 때 '어업'으로 예측한 비율: {correct_fishery_pred_fishery:.2f}%")
print(f"정답이 '어업'일 때 '임업'으로 예측한 비율: {correct_fishery_pred_forest:.2f}%")
print(f"정답이 '어업'일 때 '농업'으로 예측한 비율: {correct_fishery_pred_agriculture:.2f}%")

정답이 '임업'일 때 '임업'으로 예측한 비율: nan%
정답이 '임업'일 때 '농업'으로 예측한 비율: nan%
정답이 '임업'일 때 '어업'으로 예측한 비율: nan%
정답이 '농업'일 때 '농업'으로 예측한 비율: nan%
정답이 '농업'일 때 '임업'으로 예측한 비율: nan%
정답이 '농업'일 때 '어업'으로 예측한 비율: nan%
정답이 '어업'일 때 '어업'으로 예측한 비율: nan%
정답이 '어업'일 때 '임업'으로 예측한 비율: nan%
정답이 '어업'일 때 '농업'으로 예측한 비율: nan%


/tmp/ipython-input-2357183879.py:5: RuntimeWarning: invalid value encountered in scalar divide
  correct_forest_pred_forest = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '임업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100
/tmp/ipython-input-2357183879.py:8: RuntimeWarning: invalid value encountered in scalar divide
  correct_forest_pred_agriculture = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '농업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100
/tmp/ipython-input-2357183879.py:11: RuntimeWarning: invalid value encountered in scalar divide
  correct_forest_pred_fishery = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '어업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100
/tmp/ipython-input-2357183879.py:17: RuntimeWarning: invalid value encountered in scalar divide
  correct_agriculture_pred_agriculture = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '농업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100
/tmp/ipython-input-235

In [61]:
특허_카테고리_list = ['임업',
 '어업',
 '농업']

In [62]:
특허_카테고리_list

['임업', '어업', '농업']

In [ ]:
#기존 Prompt-Tuning전 프롬프트
system_prompt = f"너는 특허 카테고리를 분류하는 전문가야. \
아래 내용을 다음 특허 카테고리 중 하나로 분류해줘. 가능한 특허 카테고리 : {특허_카테고리_list}. \
최종 출력 결과는 다른말은 하지말고 분류한 카테고리만 출력해줘."
system_prompt

"너는 특허 카테고리를 분류하는 전문가야. 아래 내용을 다음 특허 카테고리 중 하나로 분류해줘. 가능한 특허 카테고리 : ['임업', '어업', '농업']. 최종 출력 결과는 다른말은 하지말고 분류한 카테고리만 출력해줘."

In [63]:
#기존 프롬프트에서 Prompt-Tuning을 한 ㅍ롬프트
system_prompt_ver2 = f"너는 특허 카테고리를 분류하는 전문가야. \
아래 내용을 다음 특허 카테고리 중 하나로 분류해줘. 가능한 특허 카테고리 : {특허_카테고리_list}. \
'농업' 카테고리를 '임업' 카테고리를 분류하지 않도록 주의해. '임업'은 '삼림에서 주로 나무를 벌채하고 목재를 생산하는 산업'을 의미해. \
최종 출력 결과는 다른말은 하지말고 분류한 카테고리만 출력해줘."
system_prompt_ver2

"너는 특허 카테고리를 분류하는 전문가야. 아래 내용을 다음 특허 카테고리 중 하나로 분류해줘. 가능한 특허 카테고리 : ['임업', '어업', '농업']. '농업' 카테고리를 '임업' 카테고리를 분류하지 않도록 주의해. '임업'은 '삼림에서 주로 나무를 벌채하고 목재를 생산하는 산업'을 의미해. 최종 출력 결과는 다른말은 하지말고 분류한 카테고리만 출력해줘."

In [64]:
def generate_response(system_message, user_message, tokenizer, model, max_new_token):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    outputs = model.generate(
        input_ids,
        max_new_tokens=max_new_token,
        eos_token_id=terminators,
        do_sample=False,   # 항상 가장 확률이 높은 값으로 예측
        temperature=0.6,
        top_p=0.9
    )
    response = outputs[0][input_ids.shape[-1]:]

    return tokenizer.decode(response, skip_special_tokens=True)

In [65]:
import csv

# 1. 기존 CSV 파일을 읽어서 application_number 목록을 추출
try:
    existing_df = pd.read_csv('test_output_ver2.csv', encoding='utf-8')
    existing_app_numbers = set(existing_df['application_number'].astype(str))
except FileNotFoundError:
    # 파일이 없을 경우 빈 set으로 초기화
    existing_app_numbers = set()

In [67]:
total_num = len(filtered_df)

# 2. 새 데이터를 추가할 CSV 파일을 연다
with open('test_output_ver2.csv', mode='a', newline='', encoding='utf-8') as file:
    writer = csv.writer(file, quoting=csv.QUOTE_MINIMAL)

    # 3. 데이터프레임 순회하면서 application_number가 중복되지 않으면 추가
    for i, (index, row) in enumerate(filtered_df.iterrows()):
        app_number = str(row['application_number'])  # 문자열로 변환하여 비교
        if app_number not in existing_app_numbers:
            llama3_1_inference_result = generate_response(system_message=system_prompt_ver2,
                                          user_message=row['combined_string'],
                                          tokenizer=tokenizer,
                                          model=model,
                                          max_new_token=512)

            writer.writerow([i, row['application_number'], row['combined_string'], row['Ltext'], llama3_1_inference_result])
            print(f"{i}/{total_num} Row {i}: application_number: {row['application_number']}, combined_string: {row['combined_string']}, Ltext: {row['Ltext']}, prediction: {llama3_1_inference_result}")
            existing_app_numbers.add(app_number)  # 새로 추가된 번호는 추적
        else:
            print(f"Skipping Row {i}: application_number {row['application_number']} already exists.")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


0/1150 Row 0: application_number: 1020210033985, combined_string: invention_title: 수목 보호 지지대 abstract: 본 발명은 수목 보호 지지대에 관한 것으로, 지면에 하단이 고정되는 다수의 버팀부재의 상단을 용이하게 고정시키는 본체 및 수목의 둘레 면에 접하여 지지하는 다수의 밀착부재 그리고 상기 버팀부재 및 밀착부재를 안정적으로 본체에 장착될 수 있도록 개량한 구조가 간결하게 이루어짐으로써 생산성 향상과 간편하게 설치할 수 있는 작업의 효율성을 높일 수 있고, 안정적으로 수목이 성장되도록 지지함은 물론 수목의 성장에 따른 굵기(체적)의 변화에 따른 밀착부재의 고정위치를 용이하게 조정할 수 있도록 이루어지는 수목 보호 지지대에 관한 것이다. claims: 수목(100)의 둘레 면에 접촉되는 다수의 밀착부재(2)와, 상기 밀착부재(2)를 안착시키면서 수목(100)을 중앙에 위치시키도록 일측이 개방된 중공부(11)를 갖는 본체(1)와, 상기 본체(1)의 개방부분을 폐쇄하는 고정편(3) 및 상기 본체(1)에 상단이 고정되고 하단이 지면에 고정되는 다수의 버팀부재(4)로 이루어진 수목 보호 지지대에 있어서,상기 본체(1)는 수목(100)이 위치하는 중공부(11)를 기준으로 3~4개소에 상기 버팀부재(4)의 상단이 돌출되면서 끼워지도록 천공된 다수의 끼움구멍(12)과, 이 끼움구멍(12)과 끼움구멍(12) 사이에 형성되어 상기 밀착부재(2)를 고정하는 고정볼트가 체결되는 나사공(13) 및 이탈을 방지하면서 수목(100)의 중심으로 직선 이동할 수 있도록 밀착부재(2)를 안내하는 가이드편(14)이 돌출 형성되어 이루어지고;상기 고정편(3)은 고정볼트에 의해 장착되어 본체(1)의 개방부분을 폐쇄하면서 내구성을 증대시키고 상기 밀착부재(2)를 고정하는 고정볼트가 체결되는 나사공(31) 및 이탈을 방지하면서 수목(100)의 중심으로 직선 이동할 수 있도록 밀착부재(2)를 안내하는 가이드편(32)이 돌출 형성

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


1/1150 Row 1: application_number: 2020210000760, combined_string: invention_title: 나무보호장치 abstract: 강풍에 나무들이나 전봇대들이 뽑히거나 부러지고 넘어져서 피해를 입는 사고가 많다. 이러한 문제들을 예방하고자 보호 장치를 땅속과 나무에 설치하여 피해를 입는 사고를 예방하고 미관상 문제도 해결하고자 한다. claims: '기초보호대' 와 '땅속지지대' 를 땅속에 설치하여 강풍에 나무가 옆으로 기울거나 뿌리가 뽑혀서 넘어지는 것의 위험으로부터 보호하는 장치.'추가보호대' 를 '기초보호대' 위에 연결 설치하여 나무 꼭대기까지 부러지지 않게 보호하는 장치.'뚜껑겸연결핀' 을 이용하여 '기초보호대' 와 '추가보호대' 를 연결하면 성장에 방해가 되지 않으면서 나무 전체가 보호되는 장치.'15도연결대' 를 옆으로 구부러졌거나 휘어진 나무에 설치하여 부러지는 것의 위험으로부터 보호하는 장치.청구항 1에 있어서, '기초보호대' 의 구조는 둥근 파이프 구조로 되어 있으며, 상단부는 3개, 혹은 4개의 통로가 중간부까지 관통할 수 있게 뚫려 있는데, 이 3개, 혹은 4개의 통로 역할로 인하여 '땅속지지대' 가 3각형, 혹은 4각형의 각도와 간격을 정확하게 잡혀서 땅속에 비스듬하게 사선으로 설치되므로 강풍이 어느 방향으로 불어오더라도 견딜 수 있게 된다.청구항 1에 있어서, '기초보호대' 하단부 끝부위는 땅속에 잘 들어갈 수 있게 뾰족하게 되어 있어서 이미 기존에 심어진 나무에는 땅을 파지 않고 말뚝처럼 박아서 설치할 수 있다.청구항 1과 청구항 2에 있어서, '기초보호대' 와 '땅속지지대' 의 길이와 넓이 사이즈는 '나무보호장치용도' 와 '전봇대보호장치' 두 종류가 있는데, 길이와 굵기만 다를 뿐 구조와 모양은 동일하다.청구항 3에 있어서, 뚜껑겸연결핀' 은 상단부는 봉해져 있어서 빗물 유입도 막아주면서 연결도 가능한 구조이고, 상단부의 면에는 식재할 때 나무의 이름과 연월일을 기록할 수도 있으며, 중

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


...

1146/1150 Row 1146: application_number: 1020200133158, combined_string: invention_title: 분무식 수경 재배장치 abstract: 본 발명은 본체; 식재된 육묘 뿌리가 아래로 통과하는 다수개의 뿌리통과공이 형성된 재배대가 상기 본체에 상하로 다단 배치된 트레이부; 상기 본체에 설치되어 다단 배치된 상기 재배대의 하부에 각각 배치되며 상기 재배대에 식재된 육묘 뿌리로 양액을 분무하는 분무노즐을 포함하는 복수개의 양액분무부; 상하로 다단 배치된 상기 재배대의 상부에 각각 배치되어 상기 재배대에 식재된 육묘로 하나 이상의 파장을 갖는 광을 조사하는 조명장치를 포함하는 복수개의 조명장치부; 상기 재배대의 하부에 배치되는 각각의 양액분무부의 하부에 각각 설치되며, 상기 양액분무부로부터 분무된 양액을 회수하여 일측에서 타측으로 이동시키는 복수개의 방수패드를 포함하는 양액회수부; 및 다단 배치된 상기 복수개의 방수패드의 일측에 구비되며 상기 방수패드로 회수되는 양액에 의해 상기 방수패드에 이슬이 맺히는 것을 방지하는 복수개의 송풍부를 포함하되, 상기 복수개의 방수패드는 윗변과 아랫변이 서로 다른 길이를 가지는 사다리꼴 형상을 가지고, 상기 방수패드의 양측변을 각각의 양액분무부의 하부에 설치 시 상기 방수패드의 윗변과 아랫변의 높이가 달라지며, 윗변 보다 아래쪽에 위치하는 아랫변 측으로 양액이 이동하고, 상기 송풍부가 상기 방수패드에 이슬이 맺히는 것을 방지하여 상기 방수패드에 맺혀진 이슬에 의해 양액이 식물로 필요이상으로 공급되는 것을 방지하는 것을 특징으로 하는 분무식 수경 재배장치를 제공한다.따라서, 방수패드가 사다리꼴 형상을 가짐으로써 방수패드를 양액분무부의 하부에 설치 시 방수패드로 포집되는 양액을 별도의 이동수단을 이용하지 않고 중력을 이용하여 원하는 지점으로 원활하게 이동시킬 수 있고, 이동된 양액을 순환시킴으로써 양액의 낭비를 줄일 수 있다. claims: 본체;식재된 육묘 뿌리가 아래로 통과

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


1147/1150 Row 1147: application_number: 1020200133159, combined_string: invention_title: 재배용 포트 abstract: 본 발명은 상하가 개방되고, 높이방향의 축을 따라 아래로 갈수록 단면적이 줄어드는 테이퍼진 형상을 가지며, 장기성 작물의 뿌리가 결착된 인공배지가 삽입되고, 둘레면에는 복수개의 절개홈이 상하방향으로 형성되며 장기성 작물의 뿌리가 성장 시 외측으로 벌어지는 배지결합부; 및 상기 배지결합부의 상단에서 외측으로 수평되게 연장되어 재배베드의 구멍에 걸림고정되며, 상기 재배베드의 형성된 구멍의 빈공간으로 복사선이 통과하는 것을 차단하는 차단부를 포함하되, 상기 배지결합부는 상부단이 상기 차단부와 연결되는 전개 본체와, 상기 전개 본체의 하부단과 연결되며 상기 인공배지가 안착되는 안착 본체를 포함하며, 상기 안착 본체는 복수개의 상기 절개홈에 의해 인접하는 복수개의 안착편으로 구획되고, 상기 복수개의 안착편이 장기성 작물의 뿌리가 성장 시 외측으로 벌어지며, 상기 복수개의 절개홈은 상기 안착 본체를 상하로 절개한 후 상기 전개 본체의 하부단을 절개하고, 상기 안착 본체의 하부단에는 상기 인공배지가 안착되는 안착단이 내부로 돌출되게 구비되며, 상기 안착단에 상기 인공배지가 안착되어 상기 인공배지가 상기 안착 본체의 하부로 이탈되는 것이 방지되는 것을 특징으로 하는 재배용 포트를 제공한다.따라서, 장기성 작물의 뿌리가 수용되는 안착 본체가 뿌리의 성장에 대응하여 벌어지는 구조를 가짐으로써 뿌리의 커진 부위에 경맥경화 현상이 발생하여 뿌리가 썩게 되는 것을 방지할 수 있고, 장기성 작물을 양액재배가 아닌 수경으로도 재배가 가능하도록 한다. claims: 상하가 개방되고, 높이방향의 축을 따라 아래로 갈수록 단면적이 줄어드는 테이퍼진 형상을 가지며, 장기성 작물의 뿌리가 결착된 인공배지가 삽입되고, 둘레면에는 복수개의 절개홈이 상하방향으로 형성되며 장기성 작물의 뿌리가 성장 시 외측으로 벌어

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


1148/1150 Row 1148: application_number: 1020200121174, combined_string: invention_title: 거치용 식물 재배 장치 abstract: 본 발명에 따른 거치 대상물의 외면에 거치될 수 있는 거치용 식물 재배 장치는, 거치 대상물의 외면을 따라 길이 조절 가능하게 형성되는 하우징 및 상기 하우징의 일측에 배치되는 식물이 식재되는 식재부로 구성된다. claims: 거치 대상물의 외면의 적어도 일부에 거치될 수 있는 거치용 식물 재배 장치로서,상기 거치 대상물의 외면을 따라 길이 조절 가능하게 형성되고, 사각형의 거치대상물에 거치될 수 있도록 적어도 한 부분이 'ㄱ'형상으로 절곡 형성되는 중공의 하우징; 및 식물이 상기 하우징의 내부에서 상기 하우징을 따라 생장가능하도록 상기 하우징의 일측에 배치되는 식재부;를 포함하는 거치용 식물 재배 장치.거치 대상물의 외면의 적어도 일부에 거치될 수 있는 거치용 식물 재배 장치로서,상기 거치 대상물의 외면을 따라 길이 조절 가능하게 형성되고, 사각형의 거치대상물에 거치될 수 있도록 형성되며 적어도 두 부분이 'ㄱ'형상으로 절곡 형성되는 중공의 하우징; 및 식물이 상기 하우징의 내부에서 상기 하우징을 따라 생장가능하도록 상기 하우징의 양 단에 배치되는 식재부;를 포함하는 거치용 식물 재배 장치., Ltext: 농업, prediction: '농업'
1149/1150 Row 1149: application_number: 1020200111002, combined_string: invention_title: 떡차 및 그의 제조 방법 abstract: 본 발명은, 녹차 생엽을 숙성 발효하여 떡처럼 덩어리로 만드는 고형차인 떡차의 제조 방법에 관한 것으로, 녹차 생엽의 채엽 및 선별 단계와, 시들기 위조 단계와, 찌기 단계와, 성형 단계와, 1차 자연 건조 단계와, 황토방 발효 단계와, 2차 자연 건조 단계와, 저장 단계와, 화롯불에 굽는 단계를 포함한다. claims: (

In [68]:
from google.colab import files

# 파일 다운로드
files.download('test_output_ver2.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [69]:
# CSV 파일 읽기
df = pd.read_csv('test_output_ver2.csv', header=None)

# 값이 동일한지 확인
equal_count = (df.iloc[:, 3] == df.iloc[:, 4]).sum()

# 전체 데이터에서 동일한 값이 차지하는 비율 계산
total_rows = len(df)
equal_percentage = (equal_count / total_rows) * 100

# 결과 출력
print(f"Meta-Llama-3.1-8B-Instruct 모델 예측 결과 정확도 계산('농업','임업', '어업') (프롬프트 튜닝 버전 - Ver2): {equal_percentage:.2f}%")
# Meta-Llama-3.1-8B-Instruct 모델 예측 결과 정확도 계산('농업','임업', '어업') (Base 버전): 42.68%
# Meta-Llama-3.1-8B-Instruct 모델 예측 결과 정확도 계산('농업','임업', '어업') (프롬프트 튜닝 버전 - Ver2): 70.09%

Meta-Llama-3.1-8B-Instruct 모델 예측 결과 정확도 계산('농업','임업', '어업') (프롬프트 튜닝 버전 - Ver2): 22.09%


In [70]:
# 정답이 '임업'일 때
correct_label = '임업'

# 정답이 '임업'일 때 '임업'으로 예측한 비율
correct_forest_pred_forest = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '임업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '임업'일 때 '농업'으로 예측한 비율
correct_forest_pred_agriculture = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '농업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '임업'일 때 '어업'으로 예측한 비율
correct_forest_pred_fishery = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '어업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '농업'일 때
correct_label = '농업'

# 정답이 '농업'일 때 '농업'으로 예측한 비율
correct_agriculture_pred_agriculture = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '농업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '농업'일 때 '임업'으로 예측한 비율
correct_agriculture_pred_forest = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '임업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '농업'일 때 '어업'으로 예측한 비율
correct_agriculture_pred_fishery = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '어업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '어업'일 때
correct_label = '어업'

# 정답이 '어업'일 때 '어업'으로 예측한 비율
correct_fishery_pred_fishery = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '어업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '어업'일 때 '임업'으로 예측한 비율
correct_fishery_pred_forest = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '임업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '어업'일 때 '농업'으로 예측한 비율
correct_fishery_pred_agriculture = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '농업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 결과 출력
print(f"정답이 '임업'일 때 '임업'으로 예측한 비율: {correct_forest_pred_forest:.2f}%")
print(f"정답이 '임업'일 때 '농업'으로 예측한 비율: {correct_forest_pred_agriculture:.2f}%")
print(f"정답이 '임업'일 때 '어업'으로 예측한 비율: {correct_forest_pred_fishery:.2f}%")

print(f"정답이 '농업'일 때 '농업'으로 예측한 비율: {correct_agriculture_pred_agriculture:.2f}%")
print(f"정답이 '농업'일 때 '임업'으로 예측한 비율: {correct_agriculture_pred_forest:.2f}%")
print(f"정답이 '농업'일 때 '어업'으로 예측한 비율: {correct_agriculture_pred_fishery:.2f}%")

print(f"정답이 '어업'일 때 '어업'으로 예측한 비율: {correct_fishery_pred_fishery:.2f}%")
print(f"정답이 '어업'일 때 '임업'으로 예측한 비율: {correct_fishery_pred_forest:.2f}%")
print(f"정답이 '어업'일 때 '농업'으로 예측한 비율: {correct_fishery_pred_agriculture:.2f}%")


정답이 '임업'일 때 '임업'으로 예측한 비율: 72.22%
정답이 '임업'일 때 '농업'으로 예측한 비율: 2.78%
정답이 '임업'일 때 '어업'으로 예측한 비율: 0.00%
정답이 '농업'일 때 '농업'으로 예측한 비율: 12.30%
정답이 '농업'일 때 '임업'으로 예측한 비율: 44.46%
정답이 '농업'일 때 '어업'으로 예측한 비율: 0.00%
정답이 '어업'일 때 '어업'으로 예측한 비율: 14.35%
정답이 '어업'일 때 '임업'으로 예측한 비율: 45.65%
정답이 '어업'일 때 '농업'으로 예측한 비율: 0.43%
